In [51]:
import pandas as pd
import numpy as np
from collections import Counter

In [4]:
df = pd.read_csv('qevasion_train.csv')

In [61]:
df['model_input'][0:5]

0    [Q]: How would you respond to the accusation t...
1    [Q]: Do you think President Xi is being sincer...
2    [Q]: Do you believe the country's slowdown and...
3    [Q]: Are you worried about the meeting between...
4    [Q]: Is the President's engagement with Asian ...
Name: model_input, dtype: str

# Model Wrapper

Model wrapper incapsulates the process of running a specific model

*TODO*: Add wrappers for LLMs

In [15]:
# Dummy class for model wrapper
# Eval method accepts Q&A pair from dataset and returns model CoT and evasion_label
class SampleModelWrapper:
    def  __init__(self):
        pass

    def eval(self, qa_pair: str) -> (str, str):
        return 'very complex thoghts', 'Explicit'

In [27]:
def reformat_label(label: str) -> str:
    return label.strip().capitalize()

In [17]:
wrapper = SampleModelWrapper()
wrapper.eval('Q: Question? A: Answer')

('very complex thoghts', 'Explicit')

# Label Mapping

In [28]:
def map_evasion_to_clarity(evasion_label: str) -> str:
    if evasion_label == 'Explicit':
        return 'Clear Reply'
    elif evasion_label in ['Implicit', 'Dodging', 'General', 'Deflection', 'Partial']:
        return 'Ambivalent'
    elif evasion_label in ['Declining', 'Ignorance', 'Clarification']:
        return 'Clear Non-Reply'
    else:
        raise AttributeError('unknown label ' + label)

# Voting Policy

There are 2 voting policies in the paper: majority voting and separate voting, where votes are not aggregated

In [42]:
class MajorityPolicy:
    def __init__(self, weight):
        self.w = weight

    def vote(self, labels: list[str]) -> str:
        return Counter(labels).most_common(1)[0][0]

# Deliberative Complexity Gating

In [55]:
# theta - Q_25 of 1st model response lengths computed over the current inference batch
# L - avg response length of current sample
# s - 2nd model self-consistency
def dcg(theta, L, s):
    return theta > L and s < 1.0

# Weighted Voting

In [53]:
def vote_weighted(labels_1, policy_1, labels_2, policy_2, dcg_on):
    votes = {'Clear Reply': 0, 'Ambivalent': 0, 'Clear Non-Reply': 0}
    for labels, policy in [(labels_1, policy_1), (labels_2, policy_2)]:
        if not policy is None:
            if dcg_on:
                votes['Ambivalent'] += policy.w
            else:
                votes[policy.vote(labels)] += policy.w
        else:
            for label in labels:
                votes[label] += 1
    return max(votes, key=votes.get)

# Full Pipeline

*first_model* - plays the role of Grok from paper, *second_model* - the role of Gemini

In [50]:
# TODO implement model wrappers
first_model = SampleModelWrapper()
second_model = SampleModelWrapper()

In [63]:
def run_inference(model_1, model_2, inference_batch, vote_policy_1 = None, vote_policy_2 = MajorityPolicy(4), use_dcg = True, sk_1 = 5, sk_2 = 5):
  response_lengths = []
  self_consistencies = []
  model_1_answers = []
  model_2_answers = []

  # Run inference
  for pair in inference_batch:
      ans = []
      for i in range(sk_1):
          _, l = model_1.eval(pair)
          l = reformat_label(l)
          ans.append(l)
      s = max(Counter(ans).values()) / len(ans)
      model_1_answers.append(ans)
      self_consistencies.append(s)

      ans = []
      lengths = []
      for i in range(sk_2):
          t, l = model_2.eval(pair)
          l = reformat_label(l)
          ans.append(l)
          lengths.append(len(t))
      model_2_answers.append(ans)
      response_lengths.append(np.average(lengths))

  # Voting and DCG
  final_labels = []
  theta = np.quantile(response_lengths, 0.25)
  for i in range(len(model_1_answers)):
      clarity_1 = [map_evasion_to_clarity(l) for l in model_1_answers[i]]
      clarity_2 = [map_evasion_to_clarity(l) for l in model_2_answers[i]]

      result = ""
      if use_dcg:
          o = dcg(theta, response_lengths[i], self_consistencies[i])
          result = vote_weighted(clarity_1, vote_policy_1, clarity_2, vote_policy_2, o)
      else:
          result = vote_weighted(clarity_1, vote_policy_1, clarity_2, vote_policy_2, False)
      final_labels.append(result)
  return final_labels